# R2b — Colab A100 vLLM: corrected-protocol pilot

This notebook prepares and runs the **corrected R2b** GPU pilot for *What Does a Wish Leave
Out?* on one Colab **A100** runtime. The committed `results/levels/` was rebuilt with
`make levels VISIBLE_TESTS=inputs_only`: the intent is the whole prose block and the
HumanEval+/MBPP+ visible tests print observed outputs instead of expected results. R2b
numbers replace the R2 pilot's only after the operator runs this notebook and the results are
merged; the R2 artifacts stay committed as the earlier measurement. The notebook does
**not** call a paid API and never prints a token or key.

**What it produces**

- `results/runs/r2b-pilot/smoothing/` — 500 detector-train + 100 pilot wishes smoothed by
  Qwen3-8B, with provenance and SHA-256;
- `results/detector-smoothed/` — the five-label detector refit with Qwen3-Embedding-0.6B on
  the smoothed train wishes;
- `results/runs/r2b-dry/` — a 5-task × 9-condition dry run with traces, tokens, cost and time;
- `results/runs/r2b-pilot/` — the 100-task × 3-sample × 9-condition pilot, sharded and
  resumable;
- `results/runs/r2b-pilot/summary.json`, `report.md` — pass@1, recovery, per-category
  recovery, failure buckets, tokens and GPU-hour estimate;
- a downloadable `r2b-artifacts.zip` and an optional results branch/PR.

**Before you start**

1. **Read [`COLAB.md`](../COLAB.md)** — expected wall-clock, resume procedure, the
   remote-endpoint alternative and troubleshooting.
2. Runtime → **Change runtime type → A100 GPU**. The notebook checks this and stops
   otherwise.
3. Add `GITHUB_TOKEN` (repo-scoped, read + contents write) in **Runtime → Secrets** and
   enable notebook access. Without it you can upload a repository archive instead; the
   results push is then unavailable.
4. Optionally add `HF_TOKEN`; Qwen3-8B-AWQ and Qwen3-Embedding-0.6B are public, so this is
   only for rate limits / private mirrors.
5. Budget: **~18 GPU-hours** total. The default two shards are each ~6–9 h; a Colab session
   is ~12 h, and a disconnect resumes from completed units.
6. `PERSIST_TO_DRIVE = True` snapshots artifacts to Google Drive after every shard. Restore
   is automatic on the next session; this run uses a new `superagent-r2b-clean` directory.

**Order of cells:** run top to bottom. The long steps are vLLM install (~10 min), smoothing
(~600 short calls), the detector refit (CPU) and the pilot shards. Step 3 materializes the
committed corrected levels; it does not run `make levels`.

**Known residual:** BigCodeBench visible tests are assertion-based, so their `examples` stay
visible. HumanEval+/MBPP+ visible tests are inputs-only. Do not describe the corrected
protocol as a full examples deletion.

## 0. Runtime and parameters

Everything is a parameter: change the shard count if your A100 is faster/slower, but keep
`RUN_ID`, task/sample counts and conditions stable so the run config hash stays the same and
completed units are skipped. `VISIBLE_TESTS = "inputs_only"` must match the mode the
committed levels were built with; the cell 3 check refuses a mismatch.

In [1]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
import zipfile
from datetime import datetime, timezone
from pathlib import Path

# Colab exports an inline backend for its own matplotlib; the project venv has no
# matplotlib-inline, and every project subprocess must use a headless backend.
os.environ["MPLBACKEND"] = "Agg"

REPO_SLUG = "yb2630/superagent"
BRANCH = "main"
REPO_DIR = Path("/content/superagent")

# The project is not installed as a package ([tool.uv] package = false); all
# `python -m wishrec...` commands need src/ on PYTHONPATH.
os.environ["PYTHONPATH"] = str(REPO_DIR / "src")

RUN_ID = "r2b-pilot"
DRY_RUN_ID = "r2b-dry"
PERSIST_TO_DRIVE = True
PERSIST_DIR = Path("/content/drive/MyDrive/superagent-r2b-clean")
SESSION_BUDGET_H = 10.0          # leave headroom inside Colab's ~12 h session
MIN_SHARD_MINUTES = 30
PILOT_TASKS = 100
SAMPLES = 3
WORKERS = 2                      # harness HTTP concurrency against vLLM
SHARDS = 2                       # config-stable chunks; resumable per unit
RETRAIN_R1_DETECTOR = False      # True only if you want the pre-smoothing R1 number
ENABLE_NATIVE_TOOL_CALLS = False # try vLLM's native tool parser (see Step 4)
VLLM_PIP_SPEC = "vllm>=0.8.5"
VLLM_PORT = 8000
VLLM_MAX_MODEL_LEN = 16384

# Corrected protocol parameters (R5). The committed levels must have been built with
# `make levels VISIBLE_TESTS=inputs_only`; the derived tests are materialized with the same
# mode. BigCodeBench visible tests stay assertion-based (documented residual).
VISIBLE_TESTS = "inputs_only"
LEVELS_TASKS_PATH = "results/levels/tasks.jsonl"
LEVELS_MANIFEST_PATH = "results/levels/manifest.json"
EXPECTED_LEVELS_SHA256 = "e4cf7564b38e17c040b8a37837c4757a8b5d4045791a1621331c91a15cd36436"
ALLOW_LEVELS_DRIFT = False       # True only if you intentionally use and commit a rebuilt task set
CLEAR_CACHE_FOR_DRY_RUN = True   # a fresh dry run must not replay a restored/cached result
INSTALL_VALIDATION_REQUIREMENTS = True  # BigCodeBench tests need pandas/matplotlib/... beyond uv sync
CONDITIONS = ["B1", "H0", "H1", "H2", "H3", "H4", "B2", "B3", "DISCARD"]
SESSION_START = time.time()

print("python:", sys.version.split()[0])
gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print("gpu:", gpu.stdout.strip())
GPU_NAME = gpu.stdout.strip().split(",")[0] if gpu.returncode == 0 else ""
if not GPU_NAME:
    raise RuntimeError("No GPU found. Runtime -> Change runtime type -> A100 GPU (or L4).")
A100 = "A100" in GPU_NAME
if not A100:
    if "L4" not in GPU_NAME:
        raise RuntimeError(f"Unsupported GPU {GPU_NAME}; use an A100 or L4 runtime.")
    print("warning: L4 is a 24 GB-class GPU but 2-3x slower than A100; plan for more GPU-hours.")
print("session start:", datetime.now(timezone.utc).isoformat())

python: 3.13.16
gpu: NVIDIA A100-SXM4-40GB, 40960 MiB
session start: 2026-11-23T12:45:23.916601+00:00


## 1. Token, clone and repository identity

The token is read from a Colab secret (or a hidden prompt) and passed to `git` through an
environment-backed credential helper, so it never appears in a URL, command line or output.
Clone `main`, then record the exact commit the pilot ran at. If you cannot use a token, the
archive fallback asks for a local `.zip` of the repository.

In [2]:
def run(cmd, *, cwd=None, env=None, check=True, timeout=None):
    """Run a command, show its output, and never echo secrets."""
    shown = [str(part) for part in cmd]
    print("$", " ".join(shown))
    completed = subprocess.run(
        shown,
        cwd=str(cwd) if cwd else None,
        env=env,
        text=True,
        capture_output=True,
        timeout=timeout,
    )
    if completed.stdout:
        print(completed.stdout[-12000:])
    if completed.stderr:
        print(completed.stderr[-8000:], file=sys.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed ({completed.returncode}): {' '.join(shown)}")
    return completed


def get_secret(name, prompt):
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    try:
        return getpass.getpass(prompt)
    except Exception:
        return ""


import getpass  # noqa: E402

GITHUB_TOKEN = get_secret("GITHUB_TOKEN", "GitHub repo-scoped token (blank for archive): ")
HF_TOKEN = get_secret("HF_TOKEN", "HuggingFace token (blank for public models): ")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN

if not REPO_DIR.exists():
    if GITHUB_TOKEN:
        helper = '!f() { echo username=x-access-token; echo "password=$GITHUB_TOKEN"; }; f'
        env = dict(os.environ, GITHUB_TOKEN=GITHUB_TOKEN)
        run(
            [
                "git", "-c", f"credential.helper={helper}",
                "clone", "--depth", "1", "--branch", BRANCH,
                f"https://github.com/{REPO_SLUG}.git", str(REPO_DIR),
            ],
            env=env,
            timeout=1200,
        )
    else:
        from google.colab import files  # type: ignore
        uploaded = files.upload()
        archive = next(iter(uploaded))
        REPO_DIR.mkdir(parents=True, exist_ok=True)
        shutil.copy2(archive, "/content/superagent-archive.zip")
        shutil.unpack_archive("/content/superagent-archive.zip", REPO_DIR)
else:
    print("repository already present:", REPO_DIR)

os.chdir(REPO_DIR)
COMMIT = run(["git", "rev-parse", "HEAD"]).stdout.strip()
run(["git", "config", "user.name", "R2 Colab operator"])
run(["git", "config", "user.email", "noreply@github.com"])
print("commit:", COMMIT)
print("token present:", bool(GITHUB_TOKEN), "| HF token present:", bool(HF_TOKEN))

$ git -c credential.helper=!f() { echo username=x-access-token; echo "password=$GITHUB_TOKEN"; }; f clone --depth 1 --branch main https://github.com/yb2630/superagent.git /content/superagent
$ git rev-parse HEAD
6b236eb71d2aba4d00b5ad30b3a7412ae5f23b08

$ git config user.name R2 Colab operator
$ git config user.email noreply@github.com
commit: 6b236eb71d2aba4d00b5ad30b3a7412ae5f23b08
token present: True | HF token present: True


Cloning into '/content/superagent'...



## 2. Install `uv` and sync the CPU project environment

The project environment runs the derived-test materializer, the endpoint probe, smoothing,
the detector refit and the harness. vLLM gets its own environment in Step 4 to avoid touching the pinned
project dependencies.

In [3]:
if not shutil.which("uv"):
    run(["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"], timeout=600)
    os.environ["PATH"] = f"{Path.home()}/.local/bin:{os.environ['PATH']}"
print(run(["uv", "--version"]).stdout.strip())
run(["uv", "sync", "--frozen"], cwd=REPO_DIR, timeout=1800)
print(run([".venv/bin/python", "-c", "import sys; print(sys.version)"]).stdout.strip())
if INSTALL_VALIDATION_REQUIREMENTS:
    print("installing the BigCodeBench validation libraries into the project environment")
    run(
        [
            "uv", "pip", "install", "--python", ".venv/bin/python",
            "-r", "requirements-validation.txt",
        ],
        cwd=REPO_DIR, timeout=1800,
    )
    run(
        [
            ".venv/bin/python", "-c",
            "import pandas, numpy, matplotlib, sklearn, bs4, lxml; "
            "print('validation imports ok')",
        ],
        cwd=REPO_DIR,
        env=dict(os.environ, MPLBACKEND="Agg"),
    )

$ uv --version
uv 0.13.4 (x86_64-unknown-linux-gnu)

uv 0.13.4 (x86_64-unknown-linux-gnu)
$ uv sync --frozen


Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: .venv
Prepared 71 packages in 2m 13s
Installed 71 packages in 248ms
 + annotated-doc==0.0.7
 + annotated-types==0.8.0
 + anyio==4.16.0
 + click==8.5.1
 + cloudpickle==3.1.3
 + cuda-bindings==13.5.0
 + cuda-pathfinder==1.9.1
 + cuda-toolkit==13.2.2
 + filelock==4.2.12
 + fsspec==2026.9.0
 + h11==0.16.0
 + hf-xet==1.8.0
 + httpcore2==2.16.2
 + httpx2==2.16.2
 + huggingface-hub==2.4.0
 + idna==3.20
 + iniconfig==2.3.1
 + jinja2==3.1.6
 + jiter==0.19.0
 + joblib==1.6.0
 + markdown-it-py==4.3.0
 + markupsafe==3.0.4
 + mdurl==0.1.2
 + mpmath==1.3.0
 + narwhals==2.29.0
 + networkx==3.7
 + numpy==2.5.5
 + nvidia-cublas==13.4.1.3
 + nvidia-cuda-cccl==13.3.5.0.6
 + nvidia-cuda-cupti==13.2.86
 + nvidia-cuda-nvrtc==13.2.86
 + nvidia-cuda-runtime==13.2.86
 + nvidia-cudnn-cu13==9.26.0.51
 + nvidia-cufft==12.2.0.57
 + nvidia-cufile==1.17.1.22
 + nvidia-cusolver==12.2.0.11
 + nvidia-cusparse==12.7.10.12
 + nvidia

$ .venv/bin/python -c import sys; print(sys.version)
3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]

3.12.3 (main, Aug 31 2026, 10:18:26) [GCC 13.3.0]
installing the BigCodeBench validation libraries into the project environment
$ uv pip install --python .venv/bin/python -r requirements-validation.txt


Resolved 117 packages in 2.14s
Prepared 84 packages in 1.50s
Installed 84 packages in 218ms
 + aniso8601==10.0.1
 + asgiref==3.13.0
 + beautifulsoup4==4.15.0
 + blake3==1.0.11
 + blinker==1.9.0
 + branca==0.8.2
 + certifi==2026.11.20
 + cffi==2.1.1
 + chardet==7.7.0
 + charset-normalizer==3.5.3
 + contourpy==1.4.0
 + cryptography==50.0.2
 + cssselect==1.6.0
 + cycler==0.12.1
 + defusedxml==0.7.1
 + django==6.1.3
 + et-xmlfile==2.0.0
 + faker==40.63.0
 + flask==3.1.3
 + flask-login==0.6.3
 + flask-mail==0.10.0
 + flask-restful==0.3.10
 + flask-wtf==1.3.0
 + folium==1.0.0
 + fonttools==4.68.0
 + formulaic==1.2.2
 + geographiclib==2.1
 + geopy==2.5.0
 + holidays==0.109
 + html5lib==1.1
 + imageio==2.38.2
 + interface-meta==2.0.1
 + itsdangerous==2.2.0
 + kiwisolver==1.5.1
 + lazy-loader==0.6
 + levenshtein==0.27.5
 + lxml==6.1.3
 + matplotlib==3.11.2
 + mechanize==0.4.10
 + natsort==8.4.0
 + nltk==3.10.4
 + opencv-python-headless==5.1.0.95
 + openpyxl==3.1.5
 + pandas==3.1.1
 + patsy==1.0

$ .venv/bin/python -c import pandas, numpy, matplotlib, sklearn, bs4, lxml; print('validation imports ok')
validation imports ok



## 3. Persistence helpers and data materialization

The pilot uses the **corrected** committed `results/levels/tasks.jsonl` (rebuilt by R5 with
`make levels VISIBLE_TESTS=inputs_only`); this step materializes its gitignored derived tests
with `python -m wishrec.materialize --visible-tests inputs_only` (no validation, no
re-selection). On a fresh runtime it downloads the three pinned raw revisions first and
verifies their SHA-256; it then writes the 1,100 derived task directories and checks each
hidden **and visible** test against the committed hashes. `make levels` remains the
development command that rebuilds the levels artifact; it is not used by the pilot.

If a Drive snapshot from a previous session exists, its `data/` directory is restored
instead. The hash check below still verifies every committed task; a stale R2 snapshot fails
the visible-test hashes because the HumanEval+/MBPP+ visible tests changed, and the
materializer regenerates them from the corrected levels. BigCodeBench visible tests are
unchanged and assertion-based (the documented residual).

In [5]:
import sys
sys.path.insert(0, "/content/superagent/src")
print("ok")

ok


In [6]:
SNAPSHOT = PERSIST_DIR / "r2b-results.zip"
SNAPSHOT_PATHS = ["results/runs", "results/detector-smoothed", "results/cache", "data", "logs", "configs/vllm.env"]
SESSION_START = time.time()


def mount_drive():
    if not PERSIST_TO_DRIVE:
        return False
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        PERSIST_DIR.mkdir(parents=True, exist_ok=True)
        return True
    except Exception as exc:
        print("Drive unavailable; artifacts stay on the Colab disk:", exc)
        return False


DRIVE_READY = mount_drive()


def restore_snapshot():
    if not DRIVE_READY or not SNAPSHOT.exists():
        return False
    if (REPO_DIR / "data" / "derived" / "levels").exists():
        print("data/derived already present; no snapshot restore")
        return False
    local = Path("/content/r2b-results.zip")
    shutil.copy2(SNAPSHOT, local)
    with zipfile.ZipFile(local) as archive:
        archive.extractall(REPO_DIR)
    print("restored snapshot:", SNAPSHOT)
    return True


def persist_snapshot(label=""):
    if not DRIVE_READY:
        return
    local = Path("/content/r2b-results.zip")
    with zipfile.ZipFile(local, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for relative in SNAPSHOT_PATHS:
            root = REPO_DIR / relative
            if root.is_file():
                archive.write(root, relative)
            elif root.is_dir():
                for path in root.rglob("*"):
                    if path.is_file():
                        archive.write(path, path.relative_to(REPO_DIR))
    shutil.copy2(local, SNAPSHOT)
    print(f"persisted snapshot {label}: {local.stat().st_size / 1e6:.1f} MB")


def load_done_shards():
    path = REPO_DIR / "results" / "runs" / RUN_ID / "shards.jsonl"
    if not path.exists():
        return {}
    done = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        try:
            record = json.loads(line)
            done[int(record["index"])] = record
        except (ValueError, KeyError, json.JSONDecodeError):
            continue
    return done


def mark_shard_done(index, seconds):
    path = REPO_DIR / "results" / "runs" / RUN_ID / "shards.jsonl"
    path.parent.mkdir(parents=True, exist_ok=True)
    record = {"index": index, "finished_at": datetime.now(timezone.utc).isoformat(), "duration_s": round(seconds, 1)}
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(record, sort_keys=True) + "\n")


restore_snapshot()

levels_path = REPO_DIR / LEVELS_TASKS_PATH
levels_sha = hashlib.sha256(levels_path.read_bytes()).hexdigest()
print("levels tasks.jsonl sha256:", levels_sha)
if levels_sha != EXPECTED_LEVELS_SHA256:
    if not ALLOW_LEVELS_DRIFT:
        raise RuntimeError(
            "The committed levels differ from the corrected R2b split: "
            f"{levels_sha} != {EXPECTED_LEVELS_SHA256}. Restore the committed corrected "
            "results/levels, or set ALLOW_LEVELS_DRIFT=True only after deciding to use and "
            "commit a rebuilt split."
        )
    print("WARNING: ALLOW_LEVELS_DRIFT=True; the run is on a rebuilt task set. Commit results/levels.")

levels_manifest = json.loads((REPO_DIR / LEVELS_MANIFEST_PATH).read_text(encoding="utf-8"))
parsing = levels_manifest.get("parsing") or {}
if parsing.get("visible_tests") != VISIBLE_TESTS:
    raise RuntimeError(
        "The committed levels were built with visible_tests="
        f"{parsing.get('visible_tests')!r}, but this notebook expects {VISIBLE_TESTS!r}. "
        f"Use the corrected results/levels from `make levels VISIBLE_TESTS={VISIBLE_TESTS}`."
    )
counts = levels_manifest["counts"]
if counts["kept"] != 1100 or counts["shortfall"]:
    raise RuntimeError(
        f"expected 1,100 kept tasks with no shortfall, got {counts['kept']} "
        f"(shortfall {counts['shortfall']})"
    )
print("levels parsing mode:", parsing, "| kept:", counts["kept"], "| shortfall:", counts["shortfall"])

records = [
    json.loads(line)
    for line in levels_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
if len(records) != 1100:
    raise RuntimeError(f"expected 1,100 committed tasks, found {len(records)}")


def derived_hidden_path(record):
    safe = "".join(
        character if character.isalnum() or character in "-_." else "_"
        for character in record["source_task_id"]
    )
    return REPO_DIR / "data" / "derived" / "levels" / record["source"] / safe / "hidden_test.py"


def derived_hash_ok(record, name, expected_field):
    path = derived_hidden_path(record).parent / name
    if not path.is_file():
        return False
    return hashlib.sha256(path.read_bytes()).hexdigest() == record[expected_field]


def stale_derived(records):
    return [
        record["key"]
        for record in records
        if not derived_hash_ok(record, "hidden_test.py", "hidden_test_sha256")
        or not derived_hash_ok(record, "visible_test.py", "visible_test_sha256")
    ]


stale = stale_derived(records)
print(f"derived hidden/visible hash check: {len(records) - len(stale)}/{len(records)} tasks")
if stale:
    print(f"stale or incomplete derived data for {len(stale)} tasks; materializing from the committed levels ...")
    run(
        [
            ".venv/bin/python", "-m", "wishrec.materialize",
            "--tasks", LEVELS_TASKS_PATH,
            "--visible-tests", VISIBLE_TESTS,
            "--download", "--clean",
        ],
        cwd=REPO_DIR, timeout=3600,
        env=dict(os.environ, PYTHONPATH=str(REPO_DIR / "src")),
    )
    stale = stale_derived(records)
    print(f"derived hidden/visible hash check after materialize: {len(records) - len(stale)}/{len(records)} tasks")
    if stale:
        raise RuntimeError(
            f"derived tests still missing or mismatched for {len(stale)} committed tasks, e.g. {stale[:5]}. "
            "Inspect data/raw and the materialize output; do not run the pilot on incomplete or stale data."
        )
print("levels ready:", len(records), "committed corrected tasks with derived tests")

# HumanEval+/MBPP+ must expose printed inputs-only output through the milestone tool.
from wishrec.harness.models import load_tasks
from wishrec.harness.tools import ToolSession

for source in ("humanevalplus", "mbppplus"):
    key = next(record["key"] for record in records if record["source"] == source)
    task = load_tasks(keys=[key])[0]
    with tempfile.TemporaryDirectory() as temporary:
        workspace = Path(temporary)
        (workspace / "visible_test.py").write_text(task.visible_test, encoding="utf-8")
        tool = ToolSession(workspace=workspace, source=source)
        tool.execute("write_file", {"path": "solution.py", "content": task.canonical_solution})
        message = tool.execute("run_visible_tests", {})
    if not message.startswith("Visible tests passed. Printed output:"):
        raise RuntimeError(
            f"{source}: run_visible_tests did not expose the printed inputs-only output: "
            f"{message[:300]}"
        )
    print(f"{source}: run_visible_tests printed output verified")
print("BigCodeBench visible tests remain assertion-based (known residual; examples stay visible)")
persist_snapshot("after levels")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
levels tasks.jsonl sha256: e4cf7564b38e17c040b8a37837c4757a8b5d4045791a1621331c91a15cd36436
levels parsing mode: {'visible_tests': 'inputs_only'} | kept: 1100 | shortfall: 0
derived hidden/visible hash check: 1100/1100 tasks
levels ready: 1100 committed corrected tasks with derived tests
humanevalplus: run_visible_tests printed output verified
mbppplus: run_visible_tests printed output verified
BigCodeBench visible tests remain assertion-based (known residual; examples stay visible)
persisted snapshot after levels: 29.9 MB


## 4. vLLM server on the A100

The README's R2 section uses `Qwen/Qwen3-8B-AWQ` with 16k context and 90% GPU memory. vLLM is
installed in `/content/vllm-env` so its CUDA pins cannot disturb the project environment. The
server writes `logs/vllm.log`; model download can take 5–15 min.

```bash
vllm serve Qwen/Qwen3-8B-AWQ --max-model-len 16384 --gpu-memory-utilization 0.90 --port 8000
```

**Tool transport.** The command below intentionally matches the ticket: without
`--enable-auto-tool-choice`, vLLM rejects or ignores OpenAI `tools`, and the harness must use
its JSON text fallback. The next cell probes both transports and records the working one.
Set `ENABLE_NATIVE_TOOL_CALLS = True` to add `--enable-auto-tool-choice
--tool-call-parser hermes` and prefer native calls.

In [7]:
VLLM_ENV = Path("/content/vllm-env")
HF_HOME = Path("/content/models")
HF_HOME.mkdir(parents=True, exist_ok=True)
if not (VLLM_ENV / "bin" / "vllm").exists():
    run(["uv", "venv", "--python", "3.12", str(VLLM_ENV)], timeout=600)
    run(
        ["uv", "pip", "install", "--python", str(VLLM_ENV / "bin" / "python"), VLLM_PIP_SPEC],
        timeout=3600,
    )
VLLM_VERSION = run([str(VLLM_ENV / "bin" / "vllm"), "--version"]).stdout.strip()
print(VLLM_VERSION)

$ uv venv --python 3.12 /content/vllm-env
$ uv pip install --python /content/vllm-env/bin/python vllm>=0.8.5


Using CPython 3.12.3 interpreter at: /usr/bin/python3.12
Creating virtual environment at: /content/vllm-env
Activate with: source /content/vllm-env/bin/activate

Using Python 3.12.3 environment at: /content/vllm-env
Resolved 199 packages in 1.01s
Prepared 144 packages in 1m 09s
Installed 199 packages in 342ms
 + agent-detector==2.0.0
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.7
 + annotated-types==0.8.0
 + anthropic==2.12.0
 + anyio==4.16.0
 + apache-tvm-ffi==0.1.11
 + astor==0.8.1
 + attrs==26.1.0
 + blake3==1.0.11
 + cachetools==8.0.2
 + cbor2==6.3.0
 + certifi==2026.11.20
 + cffi==2.1.1
 + charset-normalizer==3.5.3
 + click==8.5.1
 + cloudpickle==3.1.3
 + compressed-tensors==0.17.0
 + cryptography==50.0.2
 + cuda-bindings==13.5.0
 + cuda-core==1.2.2
 + cuda-pathfinder==1.9.1
 + cuda-python==13.5.0
 + cuda-tile==1.8.1
 + cuda-toolkit==13.0.3.0
 + depyf==0.20.0
 + detect-installer==0.2.1
 + dill==0.4.1
 + dnspython==2.9.2
 + docstring-parse

$ /content/vllm-env/bin/vllm --version
0.34.0

0.34.0


W1123 12:52:40.149000 2932 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1123 12:52:40.150000 2932 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1123 12:52:40.150000 2932 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
FlashInfer kernels are disabled: flashinfer-cubin is not installed and nvcc (CUDA_HOME, CUDA_PATH, PATH or /usr/local/cuda) or ninja (PATH) is missing. Set CUDA_HOME to a CUDA toolkit and put ninja on PATH, or run `vllm download-kernels`.



In [8]:
import urllib.request

VLLM_URL = f"http://127.0.0.1:{VLLM_PORT}/v1"
VLLM_LOG = REPO_DIR / "logs" / "vllm.log"
VLLM_LOG.parent.mkdir(parents=True, exist_ok=True)


def models_up(timeout=2.0):
    try:
        with urllib.request.urlopen(f"{VLLM_URL}/models", timeout=timeout) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception:
        return None


if models_up() is None:
    command = [
        str(VLLM_ENV / "bin" / "vllm"), "serve", "Qwen/Qwen3-8B-AWQ",
        "--max-model-len", str(VLLM_MAX_MODEL_LEN),
        "--gpu-memory-utilization", "0.90",
        "--port", str(VLLM_PORT),
    ]
    if ENABLE_NATIVE_TOOL_CALLS:
        command += ["--enable-auto-tool-choice", "--tool-call-parser", "hermes"]
    print("$", " ".join(command))
    log_handle = VLLM_LOG.open("a", encoding="utf-8")
    VLLM_PROCESS = subprocess.Popen(
        command, stdout=log_handle, stderr=subprocess.STDOUT,
        env=dict(os.environ, HF_HOME=str(HF_HOME)), cwd=str(REPO_DIR),
    )
    deadline = time.time() + 35 * 60
    while time.time() < deadline:
        payload = models_up(timeout=5.0)
        if payload:
            print("vLLM is up:", [item.get("id") for item in payload.get("data", [])])
            break
        if VLLM_PROCESS.poll() is not None:
            print(VLLM_LOG.read_text(encoding="utf-8", errors="replace")[-6000:])
            raise RuntimeError("vLLM exited before /v1/models came up")
        time.sleep(20)
        print("waiting for /v1/models ...", flush=True)
    else:
        print(VLLM_LOG.read_text(encoding="utf-8", errors="replace")[-6000:])
        raise RuntimeError("vLLM did not become ready within 35 minutes")
else:
    print("vLLM already running:", models_up())

$ /content/vllm-env/bin/vllm serve Qwen/Qwen3-8B-AWQ --max-model-len 16384 --gpu-memory-utilization 0.90 --port 8000
waiting for /v1/models ...
waiting for /v1/models ...
waiting for /v1/models ...
waiting for /v1/models ...
waiting for /v1/models ...
waiting for /v1/models ...
waiting for /v1/models ...
waiting for /v1/models ...
vLLM is up: ['Qwen/Qwen3-8B-AWQ']


## 5. Probe native tool calls and the text fallback

`wishrec.harness.endpoint_check` probes `/v1/models`, one native tool call and one text
protocol call, then selects the transport. It writes `results/runs/r2b-pilot/endpoint.json`
and a sourceable `configs/vllm.env` (with `VLLM_API_KEY=EMPTY`; real keys are never written).

In [9]:
endpoint_env = dict(
    os.environ,
    VLLM_BASE_URL=VLLM_URL,
    VLLM_API_KEY="EMPTY",
    VLLM_MODEL="Qwen/Qwen3-8B-AWQ",
    VLLM_TIMEOUT="300",
    VLLM_MAX_RETRIES="3",
    PYTHONPATH="src",
)
probe = run(
    [
        ".venv/bin/python", "-m", "wishrec.harness.endpoint_check",
        "--run-id", RUN_ID,
        "--output", f"results/runs/{RUN_ID}/endpoint.json",
        "--env-out", "configs/vllm.env",
    ],
    cwd=REPO_DIR, env=endpoint_env, check=False,
)
if probe.returncode != 0:
    raise RuntimeError("no usable tool transport; see the probe output and COLAB.md")
PROBE = json.loads((REPO_DIR / "results/runs" / RUN_ID / "endpoint.json").read_text())
TRANSPORT = PROBE["selected_transport"]
print("selected transport:", TRANSPORT)
for line in (REPO_DIR / "configs/vllm.env").read_text(encoding="utf-8").splitlines():
    if line and not line.startswith("#") and "=" in line:
        key, value = line.split("=", 1)
        os.environ[key] = value
os.environ["PYTHONPATH"] = "src"

$ .venv/bin/python -m wishrec.harness.endpoint_check --run-id r2b-pilot --output results/runs/r2b-pilot/endpoint.json --env-out configs/vllm.env
endpoint: http://127.0.0.1:8000/v1  model: Qwen/Qwen3-8B-AWQ
models: Qwen/Qwen3-8B-AWQ
native tool calls: failed (BadRequestError: Error code: 400 - "auto" tool choice requires --enable-auto-tool-choice and --tool-call-parser to be set)
text fallback: ok
selected transport: text
wrote results/runs/r2b-pilot/endpoint.json
wrote configs/vllm.env

selected transport: text


## 6. L3 smoothing (the step R1 could not run)

Smooth the **500 detector-train** wishes and the **100 pilot** wishes with Qwen3-8B through
the endpoint, at temperature 0. The corrected L3 is the whole parsed prose intent block (R2
used only the first prose sentence), so the smoothed wishes differ from the R2 pilot's. The
300 held-out tasks are never sent to the endpoint. The artifact is resumable; a failed call
stops the step and a rerun skips completed keys.

In [10]:
smoothing_cmd = [
    ".venv/bin/python", "-m", "wishrec.harness.smoothing",
    "--run-id", RUN_ID, "--splits", "train", "pilot", "--workers", "4",
]
run(smoothing_cmd, cwd=REPO_DIR, timeout=3600)
SMOOTHING_MANIFEST = REPO_DIR / "results" / "runs" / RUN_ID / "smoothing" / "manifest.json"
SMOOTHED = REPO_DIR / "results" / "runs" / RUN_ID / "smoothing" / "wishes.jsonl"
SMOOTHED_REL = f"results/runs/{RUN_ID}/smoothing/wishes.jsonl"
smoothing_manifest = json.loads(SMOOTHING_MANIFEST.read_text())
if not smoothing_manifest["complete"] or smoothing_manifest["tasks_completed"] < 600:
    raise RuntimeError(f"smoothing incomplete: {smoothing_manifest.get('tasks_missing')}")
print("smoothed records:", smoothing_manifest["artifact"]["records"])
print("artifact sha256:", smoothing_manifest["artifact"]["sha256"])
print("fallbacks:", smoothing_manifest["fallbacks"])
persist_snapshot("after smoothing")

$ .venv/bin/python -m wishrec.harness.smoothing --run-id r2b-pilot --splits train pilot --workers 4
R2 L3 smoothing: run_id=r2b-pilot splits={'train': 500, 'pilot': 100} workers=4
endpoint: {'base_url': 'http://127.0.0.1:8000/v1', 'model': 'Qwen/Qwen3-8B-AWQ', 'tool_mode': 'text', 'temperature': 0.7, 'top_p': 0.95, 'enable_thinking': False}
sampling: temperature=0.0, top_p=1.0
output: /content/superagent/results/runs/r2b-pilot/smoothing
smoothing [10/600] bigcodebench/BigCodeBench/1015 fallback=False
smoothing [20/600] bigcodebench/BigCodeBench/1042 fallback=False
smoothing [30/600] bigcodebench/BigCodeBench/1061 fallback=False
smoothing [40/600] bigcodebench/BigCodeBench/1090 fallback=False
smoothing [50/600] bigcodebench/BigCodeBench/1123 fallback=False
smoothing [60/600] bigcodebench/BigCodeBench/123 fallback=False
smoothing [70/600] bigcodebench/BigCodeBench/16 fallback=False
smoothing [80/600] bigcodebench/BigCodeBench/186 fallback=True
smoothing [90/600] bigcodebench/BigCodeBench

## 7. Detector refit on the smoothed wishes

The frozen encoder (`Qwen/Qwen3-Embedding-0.6B`, CPU) and the linear head are unchanged; only the
input text changes from raw L3 to the smoothed corrected wishes. The refit writes
`results/detector-smoothed/` and leaves the committed R1 artifacts untouched.

`RETRAIN_R1_DETECTOR = True` additionally runs `make detector` for the pre-smoothing
comparison and records which detector was used. The pilot itself always uses the smoothed
refit.

In [11]:
R1_DETECTOR = REPO_DIR / "results" / "detector" / "model.json"
r1_record = {"retrained": RETRAIN_R1_DETECTOR, "path": str(R1_DETECTOR.relative_to(REPO_DIR))}
if RETRAIN_R1_DETECTOR:
    run(["make", "detector"], cwd=REPO_DIR, timeout=3600)
    r1_record["retrained"] = True
r1_record["sha256"] = __import__("hashlib").sha256(R1_DETECTOR.read_bytes()).hexdigest()
(REPO_DIR / "results" / "runs" / RUN_ID / "r1_detector.json").write_text(
    json.dumps(r1_record, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print("R1 detector:", r1_record)

run(
    [
        ".venv/bin/python", "-m", "wishrec.detector.train",
        "--run-id", "r2", "--smoothed-wishes", SMOOTHED_REL,
        "--output", "results/detector-smoothed",
    ],
    cwd=REPO_DIR, timeout=5400,
)
metrics = json.loads((REPO_DIR / "results" / "detector-smoothed" / "metrics.json").read_text())
encoder = metrics["detection"]["encoder"]
print(
    "smoothed refit: macro-F1", encoder["macro_f1"],
    "| constraints AUROC", encoder["per_label"]["constraints"]["auroc"],
    "| error_behaviour AUROC", encoder["per_label"]["error_behaviour"]["auroc"],
)
print("detector L3:", metrics["training"]["l3"])
DETECTOR = REPO_DIR / "results" / "detector-smoothed" / "model.json"
DETECTOR_REL = "results/detector-smoothed/model.json"
persist_snapshot("after detector refit")

R1 detector: {'retrained': False, 'path': 'results/detector/model.json', 'sha256': 'b4038545a297fcc39058f65d8b127ae524e082917e0a4e5403cd91c2b52a76d7'}
$ .venv/bin/python -m wishrec.detector.train --run-id r2 --smoothed-wishes results/runs/r2b-pilot/smoothing/wishes.jsonl --output results/detector-smoothed
# R2 five-label silence detector

- Created: 2026-11-23T12:58:30+00:00
- Git commit: `6b236eb71d2aba4d00b5ad30b3a7412ae5f23b08` (dirty)
- Data: 500 train tasks, split {'train': 500, 'heldout': 300, 'pilot': 100, 'evolution': 200}; held-out used: 0.
- Encoder: frozen `Qwen/Qwen3-Embedding-0.6B` (revision `97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3`), 1024-d, CPU.
- Head: linear 1024->5, BCE + L2, 600 full-batch AdamW steps, seed 20261121; deployed L2=0.3.

Label: 1 when the category has content in the full L0 specification, i.e. the wish
omits information the stakeholder can supply. `error_behaviour` has 85 positives in the 500 train tasks (158 across all 1,100 kept tasks); its metrics a


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1714.53it/s]



persisted snapshot after detector refit: 31.9 MB


## 8. Five-task dry run with the real endpoint, all nine conditions

This exercises every condition, the smoothed wishes and the refit detector before spending
GPU hours on 100 tasks. It prints the per-unit traces, token totals, the `$0` cost and the
measured wall/GPU time. If the dry run fails, fix the transport or prompt lengths before the
pilot. The corrected levels' inputs-only visible tests are what the dry-run tasks load.

In [12]:
DRY_RUN_DIR = REPO_DIR / "results" / "runs" / DRY_RUN_ID
if CLEAR_CACHE_FOR_DRY_RUN:
    if DRY_RUN_DIR.exists():
        shutil.rmtree(DRY_RUN_DIR)
        print("removed restored dry-run units; the dry run will be executed against the endpoint")
    cache_dir = REPO_DIR / "results" / "cache"
    if cache_dir.exists():
        shutil.rmtree(cache_dir)
        print("cleared results/cache so the dry run cannot replay cached responses")
dry_started = time.time()
run(
    [
        ".venv/bin/python", "-m", "wishrec.harness.runner", "run",
        "--run-id", DRY_RUN_ID, "--split", "pilot", "--limit", "5",
        "--samples", "1", "--workers", str(WORKERS),
        "--conditions", *CONDITIONS,
        "--smoothed", SMOOTHED_REL, "--detector", DETECTOR_REL,
        "--output", f"results/runs/{DRY_RUN_ID}",
    ],
    cwd=REPO_DIR, timeout=3600,
)
dry_summary = json.loads((DRY_RUN_DIR / "summary.json").read_text())
dry_seconds = time.time() - dry_started
dry_units = [
    json.loads(line)
    for line in (DRY_RUN_DIR / "results.jsonl").read_text(encoding="utf-8").splitlines()
    if line.strip()
]
dry_input_tokens = sum(int(unit.get("totals", {}).get("input_tokens", 0) or 0) for unit in dry_units)
dry_output_tokens = sum(int(unit.get("totals", {}).get("output_tokens", 0) or 0) for unit in dry_units)
dry_total_tokens = dry_input_tokens + dry_output_tokens
print(f"dry-run tokens: input {dry_input_tokens:,} output {dry_output_tokens:,} total {dry_total_tokens:,}")
print("dry-run units:", dry_summary["units"], "/", dry_summary["planned_units"])
for line in (DRY_RUN_DIR / "results.jsonl").read_text(encoding="utf-8").splitlines():
    compact = json.loads(line)
    print(
        f"  {compact['condition']:8s} {compact['task_key']:36s} "
        f"sample={compact['sample_index']} pass={compact['pass']} "
        f"turns={compact['turns']} q={compact['questions']} "
        f"tokens={compact.get('totals', {}).get('total_tokens', 0)}"
    )
dry_cost = {
    "units": dry_summary["units"],
    "input_tokens": dry_input_tokens,
    "output_tokens": dry_output_tokens,
    "total_tokens": dry_total_tokens,
    "estimated_cost_usd": 0.0,
    "wall_seconds": round(dry_seconds, 1),
    "gpu_name": GPU_NAME,
    "transport": TRANSPORT,
}
(DRY_RUN_DIR / "dry_run.json").write_text(
    json.dumps(dry_cost, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print("dry-run cost/time:", dry_cost)
if dry_summary["units"] != 45:
    raise RuntimeError(f"dry run expected 45 units, got {dry_summary['units']}")
if dry_seconds < 60 or dry_total_tokens == 0:
    raise RuntimeError(
        f"the dry run finished in {dry_seconds:.1f}s with {dry_total_tokens} tokens. It was "
        "restored or replayed from cache, not executed against vLLM. Clear "
        "results/runs/<dry-run-id> and results/cache and rerun."
    )
dry_passed = sum(1 for unit in dry_units if unit.get("pass"))
module_errors = [
    unit for unit in dry_units
    if "ModuleNotFoundError" in str(unit.get("bucket", ""))
    or "ModuleNotFoundError" in str(unit.get("hidden_output_tail", ""))
]
print(f"dry-run passes: {dry_passed}/{len(dry_units)}; ModuleNotFoundError units: {len(module_errors)}")
if module_errors:
    raise RuntimeError(
        f"{len(module_errors)} dry-run units failed with ModuleNotFoundError. The project venv is "
        "missing the BigCodeBench validation libraries; run `uv pip install --python "
        ".venv/bin/python -r requirements-validation.txt` (INSTALL_VALIDATION_REQUIREMENTS) and "
        "rerun the dry run. Do not start the pilot."
    )
if dry_passed == 0:
    raise RuntimeError(
        "0/45 dry-run units passed. Do not start the pilot: inspect the failure buckets, the tool "
        "transport and the validation imports first."
    )

cleared results/cache so the dry run cannot replay cached responses
$ .venv/bin/python -m wishrec.harness.runner run --run-id r2b-dry --split pilot --limit 5 --samples 1 --workers 2 --conditions B1 H0 H1 H2 H3 H4 B2 B3 DISCARD --smoothed results/runs/r2b-pilot/smoothing/wishes.jsonl --detector results/detector-smoothed/model.json --output results/runs/r2b-dry
[1/45] B1 bigcodebench/BigCodeBench/1018 sample=0 fail turns=3 q=0
[2/45] B1 bigcodebench/BigCodeBench/1003 sample=0 PASS turns=3 q=0
[3/45] B1 bigcodebench/BigCodeBench/1046 sample=0 PASS turns=9 q=0
[4/45] B1 bigcodebench/BigCodeBench/1025 sample=0 PASS turns=3 q=0
[5/45] H0 bigcodebench/BigCodeBench/1003 sample=0 fail turns=12 q=0
[6/45] B1 bigcodebench/BigCodeBench/1064 sample=0 PASS turns=6 q=0
[7/45] H0 bigcodebench/BigCodeBench/1018 sample=0 fail turns=3 q=0
[8/45] H0 bigcodebench/BigCodeBench/1046 sample=0 PASS turns=6 q=0
[9/45] H0 bigcodebench/BigCodeBench/1064 sample=0 PASS turns=3 q=0
[10/45] H1 bigcodebench/BigCodeBen


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1662.71it/s]



## 9. The 100-task × 3-sample pilot (sharded, resumable)

The runner's `--shards/--shard` chunking keeps the config hash stable, so each shard writes
into the same run directory and already-completed units are skipped. After each shard the
notebook snapshots to Drive and appends a `shards.jsonl` marker.

**If Colab disconnects:** reconnect, rerun sections 0–7 (restore is automatic; vLLM
restarts), then rerun this cell. Completed shards and units are skipped. The session budget
guard stops before starting a shard that likely cannot finish.

In [13]:
done = load_done_shards()
done_set = set(done)
print("completed shards:", sorted(done_set))
PILOT_DIR = REPO_DIR / "results" / "runs" / RUN_ID
PILOT_DIR.mkdir(parents=True, exist_ok=True)
provenance = {
    "run_id": RUN_ID,
    "levels_tasks_sha256": levels_sha,
    "levels_manifest_sha256": hashlib.sha256(
        (REPO_DIR / LEVELS_MANIFEST_PATH).read_bytes()
    ).hexdigest(),
    "levels_parsing": parsing,
    "visible_tests": VISIBLE_TESTS,
    "allow_levels_drift": ALLOW_LEVELS_DRIFT,
    "gpu_name": GPU_NAME,
    "a100": "A100" in GPU_NAME,
    "commit": COMMIT,
    "vllm_spec": VLLM_PIP_SPEC,
    "transport": TRANSPORT,
}
(PILOT_DIR / "data_provenance.json").write_text(
    json.dumps(provenance, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print("data provenance:", provenance)
shard_times = {
    str(index): float(record.get("duration_s") or 0.0) for index, record in done.items()
}
for shard in range(SHARDS):
    if shard in done_set:
        print(f"shard {shard}: already done")
        continue
    remaining = SESSION_BUDGET_H * 3600 - (time.time() - SESSION_START)
    if remaining < MIN_SHARD_MINUTES * 60:
        print(
            f"session budget exhausted before shard {shard}; "
            "reconnect and rerun from section 0. Remaining shards: "
            f"{[item for item in range(SHARDS) if item not in done_set]}"
        )
        break
    print(f"=== shard {shard + 1}/{SHARDS} ({remaining / 3600:.1f} h left in session) ===")
    shard_started = time.time()
    run(
        [
            ".venv/bin/python", "-m", "wishrec.harness.runner", "pilot",
            "--run-id", RUN_ID, "--samples", str(SAMPLES), "--workers", str(WORKERS),
            "--shards", str(SHARDS), "--shard", str(shard),
            "--smoothed", SMOOTHED_REL, "--detector", DETECTOR_REL,
            "--output", f"results/runs/{RUN_ID}",
        ],
        cwd=REPO_DIR, timeout=11 * 3600,
    )
    shard_seconds = time.time() - shard_started
    mark_shard_done(shard, shard_seconds)
    shard_times[str(shard)] = round(shard_seconds, 1)
    persist_snapshot(f"after shard {shard}")
    summary_now = json.loads((PILOT_DIR / "summary.json").read_text())
    print(f"shard {shard} done in {shard_seconds / 3600:.2f} h; units {summary_now['units']}/{summary_now['planned_units']}")

DONE_SHARDS = set(load_done_shards())
PILOT_COMPLETE = len(DONE_SHARDS) == SHARDS
gpu_time = {
    "gpu_name": GPU_NAME,
    "shard_seconds": shard_times,
    "shard_hours": {key: round(value / 3600, 3) for key, value in shard_times.items()},
    "estimated_gpu_hours": round(sum(shard_times.values()) / 3600, 3),
    "estimated_cost_usd": 0.0,
    "note": "wall clock of the runner shards; vLLM ran on the same A100 session",
}
(PILOT_DIR / "gpu_time.json").write_text(
    json.dumps(gpu_time, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
detector_semantic_sha256 = None
if (PILOT_DIR / "manifest.json").exists():
    detector_semantic_sha256 = json.loads(
        (PILOT_DIR / "manifest.json").read_text()
    )["config"]["detector"]["sha256"]
environment = {
    "commit": COMMIT,
    "gpu": GPU_NAME,
    "vllm_version": VLLM_VERSION,
    "python": sys.version.split()[0],
    "model": os.environ.get("VLLM_MODEL", "Qwen/Qwen3-8B-AWQ"),
    "max_model_len": VLLM_MAX_MODEL_LEN,
    "native_tool_flags": ENABLE_NATIVE_TOOL_CALLS,
    "transport": TRANSPORT,
    "visible_tests": VISIBLE_TESTS,
    "smoothing_sha256": smoothing_manifest["artifact"]["sha256"],
    "detector_semantic_sha256": detector_semantic_sha256,
    "created_at": datetime.now(timezone.utc).isoformat(),
}
(PILOT_DIR / "environment.json").write_text(
    json.dumps(environment, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
print("gpu time:", gpu_time)
print("environment:", environment)
print("pilot complete:", PILOT_COMPLETE, "| shards:", sorted(DONE_SHARDS))
if not PILOT_COMPLETE:
    print("Pilot not complete yet. Reconnect, rerun sections 0-7, then rerun this cell.")

completed shards: []
data provenance: {'run_id': 'r2b-pilot', 'levels_tasks_sha256': 'e4cf7564b38e17c040b8a37837c4757a8b5d4045791a1621331c91a15cd36436', 'levels_manifest_sha256': '78278ea9851c8c6ee7a3905169992d198db18d928e3c5ac67046182b90020ac4', 'levels_parsing': {'visible_tests': 'inputs_only'}, 'visible_tests': 'inputs_only', 'allow_levels_drift': False, 'gpu_name': 'NVIDIA A100-SXM4-40GB', 'a100': True, 'commit': '6b236eb71d2aba4d00b5ad30b3a7412ae5f23b08', 'vllm_spec': 'vllm>=0.8.5', 'transport': 'text'}
=== shard 1/2 (9.8 h left in session) ===
$ .venv/bin/python -m wishrec.harness.runner pilot --run-id r2b-pilot --samples 3 --workers 2 --shards 2 --shard 0 --smoothed results/runs/r2b-pilot/smoothing/wishes.jsonl --detector results/detector-smoothed/model.json --output results/runs/r2b-pilot
le=1 fail turns=1 q=0
[1207/1350] DISCARD bigcodebench/BigCodeBench/1064 sample=0 PASS turns=9 q=3
[1208/1350] DISCARD bigcodebench/BigCodeBench/1070 sample=1 PASS turns=3 q=3
[1209/1350] DISC


Loading weights: 100%|██████████| 310/310 [00:00<00:00, 1609.01it/s]



persisted snapshot after shard 0: 34.7 MB
shard 0 done in 1.38 h; units 1350/2700
=== shard 2/2 (8.4 h left in session) ===
$ .venv/bin/python -m wishrec.harness.runner pilot --run-id r2b-pilot --samples 3 --workers 2 --shards 2 --shard 1 --smoothed results/runs/r2b-pilot/smoothing/wishes.jsonl --detector results/detector-smoothed/model.json --output results/runs/r2b-pilot
sample=1 PASS turns=3 q=3
[1208/1350] DISCARD bigcodebench/BigCodeBench/1070 sample=2 PASS turns=3 q=3
[1209/1350] DISCARD bigcodebench/BigCodeBench/1070 sample=0 PASS turns=6 q=3
[1210/1350] DISCARD bigcodebench/BigCodeBench/1097 sample=0 fail turns=1 q=2
[1211/1350] DISCARD bigcodebench/BigCodeBench/1097 sample=2 fail turns=1 q=2
[1212/1350] DISCARD bigcodebench/BigCodeBench/1117 sample=1 PASS turns=3 q=3
[1213/1350] DISCARD bigcodebench/BigCodeBench/1095 sample=1 fail turns=11 q=2
[1214/1350] DISCARD bigcodebench/BigCodeBench/1133 sample=0 PASS turns=3 q=3
[1215/1350] DISCARD bigcodebench/BigCodeBench/1133 sample=

## 10. Aggregate report and verification

Unions all completed units, checks the expected unit count and that no held-out task was
used, and writes `summary.json`/`report.md` with pass@1, recovery, per-category recovery,
failure buckets, tokens and the GPU-hour estimate. This cell also requires 2,700 units
(100 tasks × 3 samples × 9 conditions) when all shards are done.

In [14]:
report_cmd = [
    ".venv/bin/python", "-m", "wishrec.harness.report",
    f"results/runs/{RUN_ID}",
]
if PILOT_COMPLETE:
    report_cmd.append("--require-complete")
report_proc = run(report_cmd, cwd=REPO_DIR, timeout=600, check=False)
PILOT_SUMMARY = json.loads((PILOT_DIR / "summary.json").read_text())
print(
    "units:", PILOT_SUMMARY["completed_units"], "/", PILOT_SUMMARY["expected_units"],
    "| complete:", PILOT_SUMMARY["complete"],
    "| held-out used:", PILOT_SUMMARY["heldout_tasks_used"],
)
if PILOT_COMPLETE and (report_proc.returncode != 0 or not PILOT_SUMMARY["complete"]):
    raise RuntimeError("the pilot report is incomplete; inspect report.md")
if PILOT_SUMMARY["heldout_tasks_used"]:
    raise RuntimeError("a held-out task appeared in the pilot; stop and report this")
persist_snapshot("after report")

$ .venv/bin/python -m wishrec.harness.report results/runs/r2b-pilot --require-complete
# R2 GPU pilot report

- Generated: 2026-11-23T15:56:41+00:00
- Tasks: 100; conditions: 9; samples per condition: 3
- Units: 2700/2700 (complete)
- Held-out tasks used: 0
- Tokens: 19,356,531 (input 18,075,167, output 1,281,364); paid API cost: $0.00
- Wall clock: 2.9 h (2.9 GPU-h estimate); unit time 5.8 h

| condition | pass@1 | pass rate | recovery | questions | turns | cap hits |
| --- | --- | --- | --- | --- | --- | --- |
| B1 | 0.623 | 0.623 | 1.000 | 0.00 | 4.34 | 21 |
| H0 | 0.520 | 0.520 | 0.000 | 0.00 | 4.40 | 17 |
| H1 | 0.287 | 0.287 | -2.259 | 0.00 | 4.51 | 10 |
| H2 | 0.443 | 0.443 | -0.743 | 2.94 | 6.36 | 29 |
| H3 | 0.537 | 0.537 | 0.162 | 2.48 | 4.43 | 14 |
| H4 | 0.517 | 0.517 | -0.032 | 0.00 | 4.74 | 38 |
| B2 | 0.513 | 0.547 | -0.065 | 0.00 | 5.00 | 0 |
| B3 | 0.310 | 0.310 | -2.033 | 0.00 | 1.00 | 0 |
| DISCARD | 0.427 | 0.427 | -0.903 | 2.48 | 4.09 | 11 |

## Per-category recove

## 11. Download the artifacts and optionally push a results branch

The zip contains the smoothed wishes, refit detector, endpoint probe, pilot summary/report,
compact results and the full per-unit traces. It does not contain a token. Keep it as the
primary handoff; the optional branch push commits only compact artifacts (no full traces) so
the PR stays reviewable.

Set `PUSH_RESULTS = True` only after checking the report. The token is passed through an
environment-backed credential helper and never printed.

In [15]:
PUSH_RESULTS = False
ZIP_INCLUDE_TRACES = True
PUSH_BRANCH = f"r2b/colab-results-{datetime.now().strftime('%Y%m%d-%H%M')}"
ARTIFACT_DIR = REPO_DIR / "r2b-artifacts"
if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True)

def copy_tree(source: Path, destination: Path):
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    elif source.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, destination)

copy_tree(PILOT_DIR / "manifest.json", ARTIFACT_DIR / "pilot" / "manifest.json")
copy_tree(PILOT_DIR / "summary.json", ARTIFACT_DIR / "pilot" / "summary.json")
copy_tree(PILOT_DIR / "report.md", ARTIFACT_DIR / "pilot" / "report.md")
copy_tree(PILOT_DIR / "results.jsonl", ARTIFACT_DIR / "pilot" / "results.jsonl")
copy_tree(PILOT_DIR / "endpoint.json", ARTIFACT_DIR / "pilot" / "endpoint.json")
copy_tree(PILOT_DIR / "gpu_time.json", ARTIFACT_DIR / "pilot" / "gpu_time.json")
copy_tree(PILOT_DIR / "r1_detector.json", ARTIFACT_DIR / "pilot" / "r1_detector.json")
copy_tree(PILOT_DIR / "environment.json", ARTIFACT_DIR / "pilot" / "environment.json")
copy_tree(PILOT_DIR / "data_provenance.json", ARTIFACT_DIR / "pilot" / "data_provenance.json")
copy_tree(DRY_RUN_DIR, ARTIFACT_DIR / "dry-run")
copy_tree(REPO_DIR / "results" / "detector-smoothed", ARTIFACT_DIR / "detector-smoothed")
copy_tree(REPO_DIR / "results" / "runs" / RUN_ID / "smoothing", ARTIFACT_DIR / "smoothing")
copy_tree(REPO_DIR / "results" / "levels" / "manifest.json", ARTIFACT_DIR / "levels" / "manifest.json")
copy_tree(REPO_DIR / "results" / "levels" / "stats.json", ARTIFACT_DIR / "levels" / "stats.json")
copy_tree(REPO_DIR / "results" / "levels" / "tasks.jsonl", ARTIFACT_DIR / "levels" / "tasks.jsonl")

run_record = f"""# R2b corrected Colab run
- commit: {COMMIT}
- run id: {RUN_ID}
- visible tests: {VISIBLE_TESTS}
- levels parsing: {parsing}
- transport: {TRANSPORT}
- pilot complete: {PILOT_COMPLETE}
- units: {PILOT_SUMMARY['completed_units']}/{PILOT_SUMMARY['expected_units']}
- held-out used: {PILOT_SUMMARY['heldout_tasks_used']}
- GPU: {GPU_NAME}
- GPU-hours (shards): {gpu_time['estimated_gpu_hours']}
- paid API cost: $0.00
"""
(ARTIFACT_DIR / "COLAB_RUN.md").write_text(run_record, encoding="utf-8")
if GITHUB_TOKEN:
    for path in ARTIFACT_DIR.rglob("*"):
        if path.is_file() and GITHUB_TOKEN in path.read_text(encoding="utf-8", errors="ignore"):
            raise RuntimeError(f"refusing to package a file containing the token: {path}")

zip_path = Path("/content/r2b-artifacts.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
    for path in ARTIFACT_DIR.rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(REPO_DIR))
    if ZIP_INCLUDE_TRACES:
        for path in (PILOT_DIR / "units").rglob("*"):
            if path.is_file():
                archive.write(path, path.relative_to(REPO_DIR))
result = str(zip_path)
print("artifacts zip:", result, f"{zip_path.stat().st_size / 1e6:.1f} MB")
try:
    from google.colab import files  # type: ignore
    files.download(result)
except Exception:
    print("not running in Colab; zip stays at", result)

if PUSH_RESULTS:
    if not GITHUB_TOKEN:
        raise RuntimeError("PUSH_RESULTS needs GITHUB_TOKEN")
    run(["git", "checkout", "-B", PUSH_BRANCH], cwd=REPO_DIR)
    run(["git", "add", "r2b-artifacts"], cwd=REPO_DIR)
    run(
        ["git", "commit", "-m", f"R2b corrected pilot results ({RUN_ID})"],
        cwd=REPO_DIR, check=False,
    )
    helper = '!f() { echo username=x-access-token; echo "password=$GITHUB_TOKEN"; }; f'
    env = dict(os.environ, GITHUB_TOKEN=GITHUB_TOKEN)
    run(
        ["git", "-c", f"credential.helper={helper}", "push", "origin", PUSH_BRANCH],
        cwd=REPO_DIR, env=env, timeout=1800,
    )
    print(f"open a PR: https://github.com/{REPO_SLUG}/compare/{BRANCH}...{PUSH_BRANCH}?expand=1")
else:
    print("PUSH_RESULTS is False; only the downloaded zip was produced")

artifacts zip: /content/r2b-artifacts.zip 9.2 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

PUSH_RESULTS is False; only the downloaded zip was produced


## Next steps

- Keep `r2b-artifacts.zip` with the corrected pilot results; it is the operator handoff.
- Report the selected transport, the corrected `make levels VISIBLE_TESTS=inputs_only`
  outcome, whether the R1 detector was retrained, GPU-hours and any missing units in the
  PR/issue thread.
- The corrected numbers replace the R2 pilot only after this run and its results are merged;
  keep R2 and R2b numbers in separate tables.
- Do not run the 300 held-out tasks yet; they belong to the later held-out study.
- The BigCodeBench visible tests stay assertion-based (examples visible); record that as a
  protocol limitation, not a full examples deletion.
- For OOM, context-length errors, transport failures, model-download problems and the
  ngrok/cloudflared alternative, see [`COLAB.md`](../COLAB.md).